In [1]:
import os
import importlib

try:
  from google.colab import drive
  drive.mount('/content/drive')
  IN_COLAB = True
except:
  IN_COLAB = False

Mounted at /content/drive


In [2]:
if IN_COLAB:
    !pip install -q "peft==0.13.2" "transformers==4.57.3" "lm_eval==0.4.9.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 108.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 9.4 MB/s eta 0:00:00


In [ ]:
BASE_RESULTS_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments"
LOG_DIR = os.path.join(BASE_RESULTS_DIR, "logs")
MODEL_DIR = os.path.join(BASE_RESULTS_DIR, "models")
BASE_EVAL_DIR = os.path.join(BASE_RESULTS_DIR, "eval")

if IN_COLAB:
    os.makedirs(LOG_DIR, exist_ok=True)
    os.makedirs(MODEL_DIR, exist_ok=True)
    os.makedirs(BASE_EVAL_DIR, exist_ok=True)

In [ ]:
%cd /content/drive/MyDrive/TUM/Pratikum/code/sliced_rag

/content/drive/.shortcut-targets-by-id/1VRkguTVtRkllTlSID9ugy4MHwCacsE_K/Pratikum/code/sliced_rag


In [ ]:
# model = "google/gemma-3-270m-it"
model = "google/gemma-3-1b-it"
model_name = model.replace("/", "-")

In [ ]:
import json
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

def build_context(example):
    paragraphs = []

    for paragraph_title, sentences in example["context"]:
        paragraph_text = " ".join(sentences)
        paragraphs.append(f"[{paragraph_title}] {paragraph_text}")

    return "\n".join(paragraphs)

def generate_predictions(model_name, dataset_path, out_path, max_new_tokens=64, limit=None):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name).cuda()
    model.eval()

    model.generation_config.pad_token_id = tokenizer.pad_token_id

    with open(dataset_path, "r", encoding="utf-8") as f:
        examples = json.load(f)

    predictions = {
        "answer": {},
        "sp": {}
    }

    if limit is None:
        limit = len(examples)

    for example in tqdm(examples[:limit]):
        question_id  = example["_id"]
        question_text = example["question"]

        context_text = build_context(example)

        prompt = (
            "Answer the question using the context. "
            "Reply only with a short answer."
            f"Context: \n{context_text}\n\n"
            f"Question: {question_text}\n\n"
            "Answer: "
        )

        model_inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=getattr(model.config, "max_position_embeddings", None),
        ).to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

        generated_text = tokenizer.decode(
            generated_ids[0][model_inputs["input_ids"].shape[-1]:],
            skip_special_tokens=True
        )

        short_answer = generated_text.splitlines()[0].strip()

        predictions["answer"][question_id] = short_answer
        predictions["sp"][question_id] = []

    with open(out_path, "w", encoding="utf-8") as f:
          json.dump(predictions, f)

    print(f"Predictions saved to {out_path}")

In [ ]:
OUT_DIR = os.path.join(BASE_EVAL_DIR, f"{model_name}_hotpot_predictions")
os.makedirs(OUT_DIR, exist_ok=True)

generate_predictions(
    model,
    "data/hotpot_dev_distractor_v1.json",
    os.path.join(OUT_DIR, f"hotpot_predictions_{model_name}.json"),
    limit=None)

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

100%|██████████| 7405/7405 [1:46:13<00:00,  1.16it/s]

Predictions saved to /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/google-gemma-3-1b-it_hotpot_predictions/hotpot_predictions_google-gemma-3-1b-it.json
